# 365 Probabilidades - Dia #145
## Qual a probabilidade de nascerem mais gêmeos hoje do que em qualquer outra época?

**Tipo:** Demografia
**Data de publicação:** 2026-11-05
**Ferramenta:** Python
**Decisão analisada:** O que a idade da mãe e a medicina mudaram no sorteio?
**Hashtag:** #365Probabilidades #Dia145

---

### 📖 A História

Em 1895, um médico alemão chamado Dionys Hellin olhou para os registros de nascimento da
Prússia e escreveu uma regra que virou folclore de maternidade. Um parto em cada 89 é de
gêmeos. Um em 89 ao quadrado, de trigêmeos. A regra passou cento e trinta anos circulando
em livro de obstetrícia e em conversa de corredor.

O Brasil registra todos os nascimentos num sistema chamado SINASC, e cada registro diz se
a gravidez era única, dupla ou tripla, e a idade da mãe. São quase 16 milhões de
nascimentos de 2019 a 2024. Dá para conferir Hellin, e dá para ver o que a idade da mãe e
a medicina fizeram com o sorteio dele.

---

### 📚 O Conceito: duas fontes de gêmeos, duas forças

Gêmeos vêm de dois jeitos. Os idênticos, de um óvulo que se divide, acontecem numa taxa
quase constante no mundo inteiro, perto de 4 por mil partos, e ninguém sabe bem por quê.
Os fraternos, de dois óvulos liberados no mesmo ciclo, variam com a mãe: sobem com a
idade (os hormônios que estimulam a ovulação aumentam até os 35 ou 40 anos), com o número
de filhos anteriores, com a genética da família e, desde os anos 1980, com a reprodução
assistida, que costuma transferir mais de um embrião.

Por isso a taxa de gêmeos de um país é um retrato de duas forças ao mesmo tempo. A
fecundidade caindo puxa a taxa para baixo, porque menos mulheres chegam à terceira ou
quarta gravidez. A maternidade mais tardia e a fertilização assistida puxam para cima. O
que o mundo viu nas últimas quatro décadas foi a segunda força ganhando.

A conta do dia é uma binomial. Cada parto é um sorteio com probabilidade p de gêmeos, p
depende da idade da mãe, e o modelo estima cada p com o intervalo de Jeffreys.

---

### 🧮 O Modelo

Cinco camadas.

1. **Partos, não bebês.** O SINASC conta nascidos; cada parto de gêmeos aparece duas
   vezes. A conversão para partos e a taxa brasileira com intervalo.
2. **Hellin contra o Brasil.** A regra de 1895 e o número de 2019-2024.
3. **Por idade da mãe.** A taxa em cada faixa, com IC de Jeffreys, e a razão entre a mãe
   de 45 e a adolescente.
4. **O Brasil no mundo.** A taxa brasileira ao lado das de Monden, Pison e Smits, 1980-85
   e 2010-15.
5. **Cem mães.** Numa sala com cem mulheres grávidas da mesma idade, a chance de pelo
   menos uma ter gêmeos.

**Tabela de fontes**

| Fonte | O que é | Número usado |
|---|---|---|
| SINASC 2019-2024 (OpenDataSUS, microdados agregados no #136) | registro de nascimentos do Brasil | 15.734.512 nascidos com tipo de gravidez e idade da mãe; 342.538 nascidos de gravidez dupla, 7.285 de tripla |
| Monden, C., Pison, G. & Smits, J. (2021). Twin Peaks: more twinning in humans than ever before. *Human Reproduction*, 36(6), 1666-1673 | compilação de 165 países | taxa mundial de 9,1 (1980-85) para 12,0 (2010-15) partos gemelares por mil; 1,6 milhão de pares por ano; África mais alta, Ásia e América do Sul mais baixas |
| Hellin, D. (1895). *Die Ursache der Multiparität der uniparen Tiere überhaupt und der Zwillingsschwangerschaft beim Menschen insbesondere* | a regra clássica | 1 parto em 89 é de gêmeos; 1 em 89² de trigêmeos |

Leitura: SINASC contado nos microdados; Monden lido no resumo (PubMed, 27/10/2026);
Hellin citado pela regra, que a literatura repete, não pelo original.

**Nota metodológica.** ×0,80 não se aplica: registro de nascimento. "Gravidez" no SINASC
é o campo GRAVIDEZ (1 única, 2 dupla, 3 tripla ou mais); registros com idade da mãe ou
tipo ignorados ficam fora e são contados. Cordas: #136 (lua cheia, mesma base) e #147
(aniversário, mesma base).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

SEMENTE = 42
rng = np.random.default_rng(SEMENTE)

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'

def mil(n):
    return f"{int(round(n)):,}".replace(",", ".")

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

def pct(x, casas=0):
    return vir(100 * x, casas) + "%"

def jeffreys(k, n):
    d = stats.beta(k + 0.5, n - k + 0.5)
    return d.mean(), d.interval(0.95)

print("✅ Bibliotecas carregadas · semente", SEMENTE)

✅ Bibliotecas carregadas · semente 42


In [2]:
# --- DADOS DA LITERATURA ---

# SINASC 2019-2024, agregado por ano, idade da mãe e tipo de gravidez (1 única, 2 dupla, 3 tripla ou mais)
G = pd.read_csv("dia-145-sinasc-gemeos-2019-2024.csv", dtype={"idade_mae": str, "tipo": str})
G = G[G["tipo"].isin(["1", "2", "3"]) & G["idade_mae"].str.fullmatch(r"\d+")].copy()
G["idade_mae"] = G["idade_mae"].astype(int)
N_IGNORADOS = int(pd.read_csv("dia-145-sinasc-gemeos-2019-2024.csv", dtype=str).query("tipo in ['x','9'] or ~idade_mae.str.match(r'^\\d+$')")["n"].astype(int).sum())

# Monden, Pison & Smits 2021, Human Reproduction 36(6)
MUNDO = {"1980-85": 9.1, "2010-15": 12.0}      # partos gemelares por mil partos
PARES_POR_ANO = 1_600_000
PAISES = 165
HELLIN = 1 / 89                                 # regra de 1895

FAIXAS = [(10, 19, "até 19"), (20, 24, "20 a 24"), (25, 29, "25 a 29"), (30, 34, "30 a 34"),
          (35, 39, "35 a 39"), (40, 44, "40 a 44"), (45, 60, "45 ou mais")]

def partos(df):
    # nascidos → partos: dupla aparece 2 vezes, tripla 3 vezes (ou mais; aproximação declarada)
    s = df.groupby("tipo")["n"].sum().reindex(["1", "2", "3"]).fillna(0)
    return {"unicos": s["1"], "duplos": s["2"] / 2, "triplos": s["3"] / 3, "nascidos": s.sum()}

T = partos(G)
T["total"] = T["unicos"] + T["duplos"] + T["triplos"]

print("=" * 70)
print("  DADOS · GÊMEOS NO BRASIL E NO MUNDO")
print("=" * 70)
print(f"\n  SINASC 2019-2024: {mil(T['nascidos'])} nascidos com tipo de gravidez e idade da mãe ({mil(N_IGNORADOS)} ignorados, fora)")
print(f"  → {mil(G[G.tipo == '2'].n.sum())} nascidos de gravidez dupla, {mil(G[G.tipo == '3'].n.sum())} de tripla ou mais")
print(f"  → em partos: {mil(T['total'])} partos; {mil(T['duplos'])} de gêmeos; {mil(T['triplos'])} de trigêmeos")
print(f"\n  Mundo (Monden, Pison & Smits 2021, {PAISES} países): {vir(MUNDO['1980-85'])} → {vir(MUNDO['2010-15'])} por mil partos; {mil(PARES_POR_ANO)} pares por ano")
print(f"  Hellin (1895): 1 em 89 = {vir(HELLIN * 1000)} por mil")
print("=" * 70)

  DADOS · GÊMEOS NO BRASIL E NO MUNDO

  SINASC 2019-2024: 15.734.512 nascidos com tipo de gravidez e idade da mãe (10.703 ignorados, fora)
  → 342.537 nascidos de gravidez dupla, 7.285 de tripla ou mais
  → em partos: 15.558.387 partos; 171.268 de gêmeos; 2.428 de trigêmeos

  Mundo (Monden, Pison & Smits 2021, 165 países): 9,1 → 12,0 por mil partos; 1.600.000 pares por ano
  Hellin (1895): 1 em 89 = 11,2 por mil


In [3]:
# --- O MODELO ---

# CAMADA 1 · a taxa brasileira, em partos
k, n = T["duplos"], T["total"]
p_br, ic_br = jeffreys(k, n)
print("=" * 70)
print("  CAMADA 1 · PARTOS, NÃO BEBÊS")
print("=" * 70)
print(f"\n  gêmeos por mil partos no Brasil 2019-2024: {vir(p_br * 1000)}  IC 95% [{vir(ic_br[0] * 1000)}; {vir(ic_br[1] * 1000)}]")
print(f"  um parto em cada {vir(1 / p_br, 0)}")
p_tri, _ = jeffreys(T["triplos"], n)
print(f"  trigêmeos: {vir(p_tri * 1000, 2)} por mil, um em cada {mil(1 / p_tri)}")
print(f"  se a conta fosse por bebê, e não por parto, daria {vir(G[G.tipo == '2'].n.sum() / T['nascidos'] * 1000)} por mil, o dobro: é o erro mais comum com esse dado")

# CAMADA 2 · Hellin
print("\n" + "=" * 70)
print("  CAMADA 2 · HELLIN CONTRA O BRASIL")
print("=" * 70)
print(f"\n  Hellin: {vir(HELLIN * 1000)} por mil (1 em 89) · Brasil: {vir(p_br * 1000)} por mil (1 em {vir(1 / p_br, 0)})")
print(f"  trigêmeos por Hellin: 1 em 89² = 1 em {89**2} · Brasil: 1 em {mil(1 / p_tri)}")
print("  a regra de 1895 bate com o Brasil de 2024 por coincidência de duas forças, como a camada 3 mostra")

# CAMADA 3 · por idade da mãe
linhas = []
for a, b, rot in FAIXAS:
    sub = G[(G.idade_mae >= a) & (G.idade_mae <= b)]
    t = partos(sub); tot = t["unicos"] + t["duplos"] + t["triplos"]
    p, ic = jeffreys(t["duplos"], tot)
    linhas.append((rot, int(tot), p, ic[0], ic[1]))
IDADE = pd.DataFrame(linhas, columns=["faixa", "partos", "p", "lo", "hi"])
razao = IDADE.p.iloc[-1] / IDADE.p.iloc[0]
print("\n" + "=" * 70)
print("  CAMADA 3 · POR IDADE DA MÃE (gêmeos por mil partos)")
print("=" * 70)
for _, r in IDADE.iterrows():
    print(f"  {r.faixa:<11s} {mil(r.partos):>11s} partos  {vir(r.p * 1000):>5s}  [{vir(r.lo * 1000)}; {vir(r.hi * 1000)}]")
print(f"\n  mãe de 45 ou mais contra mãe adolescente: {vir(razao)} vezes")
print(f"  a curva sobe até os 35-39 pelo caminho natural (ovulação) e salta aos 45 pela reprodução assistida")

# por ano
ANO = []
for y, sub in G.groupby("ano"):
    t = partos(sub); tot = t["unicos"] + t["duplos"] + t["triplos"]
    p, ic = jeffreys(t["duplos"], tot); ANO.append((int(y), int(tot), p, ic[0], ic[1]))
ANO = pd.DataFrame(ANO, columns=["ano", "partos", "p", "lo", "hi"])
incl = np.polyfit(ANO.ano, ANO.p * 1000, 1)[0]
print(f"\n  por ano: " + " · ".join(f"{int(r.ano)} {vir(r.p * 1000)}" for _, r in ANO.iterrows()))
print(f"  tendência: +{vir(incl, 2)} por mil ao ano")

# CAMADA 4 · o Brasil no mundo
print("\n" + "=" * 70)
print("  CAMADA 4 · O BRASIL NO MUNDO")
print("=" * 70)
print(f"\n  mundo 1980-85: {vir(MUNDO['1980-85'])} · mundo 2010-15: {vir(MUNDO['2010-15'])} · Brasil 2019-24: {vir(p_br * 1000)} · Hellin 1895: {vir(HELLIN * 1000)}")
print(f"  o mundo subiu {pct(MUNDO['2010-15'] / MUNDO['1980-85'] - 1)} em três décadas; o Brasil está {pct(1 - p_br * 1000 / MUNDO['2010-15'])} abaixo da média mundial de 2010-15 e subindo")

# CAMADA 5 · cem mães
print("\n" + "=" * 70)
print("  CAMADA 5 · CEM MÃES DA MESMA IDADE")
print("=" * 70)
for _, r in IDADE.iterrows():
    p_pelo_menos_um = 1 - (1 - r.p) ** 100
    esperado = 100 * r.p
    print(f"  {r.faixa:<11s} chance de pelo menos um par de gêmeos: {pct(p_pelo_menos_um)}  (esperado {vir(esperado)} pares)")
print("=" * 70)

  CAMADA 1 · PARTOS, NÃO BEBÊS

  gêmeos por mil partos no Brasil 2019-2024: 11,0  IC 95% [11,0; 11,1]
  um parto em cada 91
  trigêmeos: 0,16 por mil, um em cada 6.406
  se a conta fosse por bebê, e não por parto, daria 21,8 por mil, o dobro: é o erro mais comum com esse dado

  CAMADA 2 · HELLIN CONTRA O BRASIL

  Hellin: 11,2 por mil (1 em 89) · Brasil: 11,0 por mil (1 em 91)
  trigêmeos por Hellin: 1 em 89² = 1 em 7921 · Brasil: 1 em 6.406
  a regra de 1895 bate com o Brasil de 2024 por coincidência de duas forças, como a camada 3 mostra

  CAMADA 3 · POR IDADE DA MÃE (gêmeos por mil partos)
  até 19        2.043.932 partos    5,9  [5,8; 6,0]
  20 a 24       3.771.342 partos    8,2  [8,1; 8,3]
  25 a 29       3.846.121 partos   10,7  [10,6; 10,8]
  30 a 34       3.241.845 partos   13,7  [13,5; 13,8]
  35 a 39       2.048.017 partos   16,2  [16,0; 16,4]
  40 a 44         571.445 partos   14,8  [14,5; 15,1]
  45 ou mais       35.550 partos   30,6  [28,8; 32,4]

  mãe de 45 ou mais co

In [4]:
# --- VISUALIZAÇÃO ---
RODAPE = 'Fontes: SINASC 2019-2024 · Monden, Pison & Smits (2021), Human Reproduction · Hellin (1895) | #365Probabilidades'

# GRÁFICO 1 · o Brasil no mundo
fig1, ax1 = plt.subplots(figsize=(12, 7))
rot = ['Hellin\n1895', 'mundo\n1980-85', 'Brasil\n2019-24', 'mundo\n2010-15']
val = [HELLIN * 1000, MUNDO['1980-85'], p_br * 1000, MUNDO['2010-15']]
cor = [CINZA, DOURADO, VERMELHO, VERDE]
ax1.bar(rot, val, color=cor, alpha=0.9, width=0.6)
for i, v in enumerate(val):
    ax1.text(i, v + 0.2, vir(v), ha='center', fontsize=16, fontweight='bold')
ax1.set_ylim(0, 14); ax1.set_ylabel('partos de gêmeos a cada mil partos', fontsize=12)
ax1.set_title(f'O mundo subiu de 9,1 para 12,0 gêmeos por mil partos em três décadas; o Brasil está em {vir(p_br * 1000)}\n'
              'a regra de 1895 (1 em 89) cai em cima do Brasil de hoje por coincidência de duas forças', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-145-grafico-01-o-brasil-no-mundo.png', dpi=150, bbox_inches='tight'); plt.close()
print("✅ Gráfico 1 salvo!")

# GRÁFICO 2 · por idade da mãe
fig2, ax2 = plt.subplots(figsize=(12, 7))
x = np.arange(len(IDADE))
ax2.bar(x, IDADE.p * 1000, color=[VERDE] * 5 + [DOURADO, VERMELHO], alpha=0.9, width=0.65)
ax2.errorbar(x, IDADE.p * 1000, yerr=[(IDADE.p - IDADE.lo) * 1000, (IDADE.hi - IDADE.p) * 1000], fmt='none', ecolor='#333', capsize=6, lw=1.5)
for i, r in IDADE.iterrows():
    ax2.text(i, r.hi * 1000 + 1.2, vir(r.p * 1000), ha='center', fontsize=14, fontweight='bold')
ax2.axhline(HELLIN * 1000, color=CINZA, lw=1.5, ls='--'); ax2.text(-0.3, HELLIN * 1000 + 0.8, 'Hellin, 1 em 89', ha='left', fontsize=10.5, color=CINZA)
ax2.set_xticks(x); ax2.set_xticklabels(IDADE.faixa, fontsize=12); ax2.set_xlabel('idade da mãe', fontsize=12)
ax2.set_ylim(0, 36); ax2.set_ylabel('partos de gêmeos a cada mil partos', fontsize=12)
ax2.set_title(f'A chance de gêmeos sobe com a idade da mãe e salta depois dos 45: {vir(razao)} vezes a da adolescente\n'
              'até os 39 é a ovulação; aos 45 é a reprodução assistida aparecendo no registro', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-145-grafico-02-idade-da-mae.png', dpi=150, bbox_inches='tight'); plt.close()
print("✅ Gráfico 2 salvo!")

# GRÁFICO 3 · ano a ano
fig3, ax3 = plt.subplots(figsize=(12, 6.5))
ax3.errorbar(ANO.ano, ANO.p * 1000, yerr=[(ANO.p - ANO.lo) * 1000, (ANO.hi - ANO.p) * 1000], color=VERMELHO, lw=3, marker='o', markersize=10, capsize=5)
for _, r in ANO.iterrows():
    ax3.text(r.ano, r.p * 1000 + 0.12, vir(r.p * 1000, 2), ha='center', fontsize=12, fontweight='bold')
ax3.set_xticks(ANO.ano); ax3.set_ylim(10, 12); ax3.set_ylabel('partos de gêmeos a cada mil partos', fontsize=12)
ax3.set_title(f'Seis anos no Brasil: de {vir(ANO.p.iloc[0] * 1000, 2)} para {vir(ANO.p.iloc[-1] * 1000, 2)} por mil, {vir(incl, 2)} a mais por ano\n'
              'devagar e numa direção só; os intervalos são estreitos porque são 2,5 milhões de partos por ano', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-145-grafico-03-ano-a-ano.png', dpi=150, bbox_inches='tight'); plt.close()
print("✅ Gráfico 3 salvo!")

# GRÁFICO 4 · cem mães
fig4, ax4 = plt.subplots(figsize=(12, 6.5))
pm = 1 - (1 - IDADE.p) ** 100
ax4.bar(x, pm * 100, color=[VERDE] * 5 + [DOURADO, VERMELHO], alpha=0.9, width=0.65)
for i, v in enumerate(pm):
    ax4.text(i, v * 100 + 1.5, pct(v), ha='center', fontsize=14, fontweight='bold')
ax4.set_xticks(x); ax4.set_xticklabels(IDADE.faixa, fontsize=12); ax4.set_xlabel('idade das cem mães', fontsize=12)
ax4.set_ylim(0, 108); ax4.set_ylabel('chance de pelo menos um par de gêmeos (%)', fontsize=12)
ax4.set_title('Cem grávidas da mesma idade numa sala: a chance de pelo menos uma ter gêmeos\n'
              f'{pct(pm.iloc[0])} entre adolescentes, {pct(pm.iloc[-1])} depois dos 45', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-145-grafico-04-cem-maes.png', dpi=150, bbox_inches='tight'); plt.close()
print("✅ Gráfico 4 salvo!")

✅ Gráfico 1 salvo!
✅ Gráfico 2 salvo!
✅ Gráfico 3 salvo!
✅ Gráfico 4 salvo!


### 💡 O Insight

**Nascem mais gêmeos hoje do que em qualquer outra época. No mundo, a taxa subiu um terço
desde os anos 1980, de 9,1 para 12,0 partos gemelares por mil. No Brasil, são 11,0 por
mil, um parto em cada 91, e a taxa sobe um pouco a cada ano.**

A regra de Hellin, de 1895, dizia 1 em 89, e o Brasil de hoje cai em cima dela. Não é
porque nada mudou. É porque duas coisas mudaram em direções opostas. A fecundidade caiu, e
com ela a chance natural de gêmeos; a maternidade ficou mais tardia e a reprodução
assistida chegou, e as duas puxaram de volta. A regra acerta por acaso.

O que a idade da mãe faz com o sorteio é a parte que mais surpreende. De 6 por mil nas
adolescentes a 16 por mil aos 35-39, pelo caminho natural dos hormônios. E 31 por mil
depois dos 45, cinco vezes a adolescente, que é a fertilização assistida aparecendo no
registro de nascimento.

O que isso não diz é que ter gêmeos virou comum. Numa sala com cem grávidas de 30 a 34
anos, a chance de pelo menos uma ter gêmeos é de 75%, mas para cada uma delas continua
sendo um em 73. Raro para a pessoa, frequente para a sala.

*Você conhece gêmeos? De que idade era a mãe?*

---

### ⚠️ Limitações

**Dos dados**
- O SINASC registra a gravidez como única, dupla ou "tripla ou mais"; a conversão de
  nascidos para partos divide por 2 e por 3, e os poucos partos de quatro ou mais ficam
  levemente subcontados.
- Nascidos vivos apenas; gestações gemelares que terminam sem nascido vivo não entram.
- A idade da mãe é a declarada na DNV; registros com idade ou tipo ignorados ficaram fora
  e estão contados.
- Monden, Pison e Smits foram lidos no resumo; a taxa do Brasil no artigo deles, se
  existir, não foi conferida, e por isso o dia compara o Brasil do SINASC com a média
  mundial deles, não com o Brasil deles.

**Do modelo**
- Cada parto é tratado como um sorteio independente com a mesma chance dentro da faixa de
  idade; irmãs de gêmeos e mães que já tiveram gêmeos têm chance maior, e o modelo não
  distingue.
- A leitura "aos 45 é a reprodução assistida" é a explicação da literatura, não algo que o
  SINASC meça; o registro não diz se houve tratamento.
- A tendência de seis anos é uma reta ajustada a seis pontos; serve para dizer a direção,
  não para projetar.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*